# Method 2 — Run 0: Pre-flight (không train)

Cổng fail-closed trước mọi job training: artefact, SHA-256, split overlap, version, GPU. 0 giờ GPU training.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('bf16 supported:', torch.cuda.is_bf16_supported())
# Kỳ vọng: Tesla T4, ~15.0 GB free, bf16 = False → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: mount code + data =====
# Đẩy repo và data lên Kaggle Dataset (private) trước.
!cp -r /kaggle/input/toolcalling-vi-src/src /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-src/configs /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-data/data /kaggle/working/data
%cd /kaggle/working

import json, os, glob, sys
sys.path.insert(0, '/kaggle/working')

# Cache model HF thành Kaggle Dataset để không tải lại mỗi session.
os.environ.setdefault('HF_HOME', '/kaggle/input/hf-cache')

manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('snapshot commit:', manifest.get('git_commit'))


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [ ]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


In [ ]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


## Kết thúc Run 0

Run 0 **không train gì cả** — chỉ xác nhận package, artefact và split đúng
như đã kiểm định ở local. Đạt hết thì thoát, sang notebook Bi-Encoder chạy
Run 1 (smoke) rồi Run 2 (full).


In [ ]:
import pprint

for check in preflight['checks']:
    mark = 'PASS' if check['passed'] else 'FAIL'
    detail = check['detail']
    suffix = f' — {detail}' if detail else ''
    print('[' + mark + '] ' + check['name'] + suffix)

print()
pprint.pprint(json.load(open('data/method2/manifest.json', encoding='utf-8'))['derived'])


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/preflight_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/preflight_run.tar.gz
